# 10_Doo — Random Forest 하이퍼파라미터 튜닝

## 실험 원칙
Train 내부 5-fold 교차검증으로 튜닝하고 Validation에서 최종 비교한다. Test 데이터는 사용하지 않으며 `random_state=42`를 유지한다.

## 1. 데이터 확인
팀 전처리 결과의 Train/Validation 데이터를 사용한다. 타겟은 `churn`이며 `1`은 고객 이탈을 의미한다.

In [2]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

root = Path.cwd()
while not (root / 'data' / 'preprocessed').exists():
    if root.parent == root:
        raise FileNotFoundError('data/preprocessed 폴더를 찾을 수 없음')
    root = root.parent

data_dir = root / 'data' / 'preprocessed'
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')['churn']
X_val = pd.read_csv(data_dir / 'X_val.csv')
y_val = pd.read_csv(data_dir / 'y_val.csv')['churn']
print('X_train:', X_train.shape, '| X_val:', X_val.shape)


def evaluate(model, X, y):
    p = model.predict(X)
    q = model.predict_proba(X)[:, 1]
    return {'accuracy': accuracy_score(y, p), 'recall': recall_score(y, p),
            'precision': precision_score(y, p, zero_division=0), 'f1': f1_score(y, p, zero_division=0),
            'roc_auc': roc_auc_score(y, q)}

X_train: (2592, 10) | X_val: (864, 10)


## 2. 베이스라인 모델
기본 Random Forest를 기준선으로 학습한다. 여러 결정트리를 결합해 비선형 관계와 변수 상호작용을 표현한다.

## 3. 하이퍼파라미터 튜닝
트리 개수, 최대 깊이, 분할 조건, 리프 크기, 피처 샘플 비율과 클래스 가중치를 Train 내부 5-fold Stratified CV에서 ROC-AUC 기준으로 탐색한다.

In [3]:
baseline = RandomForestClassifier(
    n_estimators=400, max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=1,
)
params = {
    'n_estimators': [200, 400, 600, 800],
    'max_depth': [None, 5, 8, 12, 16],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced', 'balanced_subsample'],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), params, n_iter=25, scoring='roc_auc',
                            cv=cv, random_state=42, n_jobs=1, return_train_score=True)
search.fit(X_train, y_train);
tuned = search.best_estimator_
print('Best CV ROC-AUC:', round(search.best_score_, 4));
print('Best parameters:');
display(pd.Series(search.best_params_))
baseline.fit(X_train, y_train)
rows = []
for name, model in [('Baseline', baseline), ('Tuned', tuned)]:
    tr = evaluate(model, X_train, y_train);
    va = evaluate(model, X_val, y_val)
    rows.append(
        {'model': name, 'train_accuracy': tr['accuracy'], 'val_accuracy': va['accuracy'], 'recall': va['recall'],
         'precision': va['precision'], 'f1': va['f1'], 'roc_auc': va['roc_auc'],
         'train_val_gap': tr['accuracy'] - va['accuracy']})
comparison = pd.DataFrame(rows).set_index('model');
display(comparison.round(3))


Best CV ROC-AUC: 0.7769
Best parameters:


n_estimators                        400
min_samples_split                     2
min_samples_leaf                      8
max_features                       sqrt
max_depth                             8
class_weight         balanced_subsample
dtype: object

,train_accuracy,val_accuracy,recall,precision,f1,roc_auc,train_val_gap
model,,,,,,,
Baseline,1.000,0.674,0.681,0.666,0.674,0.741,0.326
Tuned,0.776,0.700,0.738,0.682,0.709,0.762,0.076


## 4. 임계값 비교
기본값 0.50 외의 임계값을 비교한다. Recall 0.80 이상 후보 중 F1-score가 가장 높은 값을 선택한다.

In [4]:
tuned_proba = tuned.predict_proba(X_val)[:, 1];
threshold_rows = []
for threshold in np.arange(0.25, 0.71, 0.05):
    pred = (tuned_proba >= threshold).astype(int)
    threshold_rows.append({'threshold': threshold, 'recall': recall_score(y_val, pred),
                           'precision': precision_score(y_val, pred, zero_division=0),
                           'f1': f1_score(y_val, pred, zero_division=0), 'predicted_churn_count': int(pred.sum())})
threshold_df = pd.DataFrame(threshold_rows);
display(threshold_df.round(3))
candidates = threshold_df[threshold_df['recall'] >= 0.80]
selected = candidates.loc[candidates['f1'].idxmax()] if not candidates.empty else threshold_df.loc[
    threshold_df['f1'].idxmax()]
print('선택 임계값');
display(selected.to_frame('selected_value'))


,threshold,recall,precision,f1,predicted_churn_count
0,0.25,0.939,0.591,0.726,678
1,0.30,0.913,0.613,0.734,636
2,0.35,0.876,0.629,0.732,595
3,0.40,0.834,0.640,0.724,556
4,0.45,0.780,0.654,0.712,509
5,0.50,0.738,0.682,0.709,462
6,0.55,0.670,0.692,0.681,413
7,0.60,0.602,0.720,0.656,357
8,0.65,0.480,0.730,0.579,281
9,0.70,0.363,0.752,0.490,206


선택 임계값


,selected_value
threshold,0.300000
recall,0.913349
precision,0.613208
f1,0.733772
predicted_churn_count,636.000000


## 5. 결과 해석과 다음 단계
튜닝 전후 Validation 성능과 Train-Validation gap을 확인한다. 최종 모델을 합의한 후 Train+Validation 재학습과 Test 1회 평가를 진행한다.